<a href="https://colab.research.google.com/github/armandobrunner92-cloud/MiPrimerAppAndroid/blob/main/app_pdf_resumen_bd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Procesamiento de PDF
1. Crea la base de datos (SQLite) con la tabla de PDF (nombre de archivo y resumen).
2. Procesa PDF (extrae el texto, genera un resumen y estadísticas) y guarda el análisis en la base de datos.

Ejecuta las celdas en orden. No necesita GPU ni modelos de IA.

In [ ]:
!pip -q install gradio pypdf fpdf2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.3/341.3 kB 11.0 MB/s eta 0:00:00


## a. Base de datos y tabla de PDF

In [ ]:
import sqlite3
import pandas as pd

DB = "pdfs.db"

def conectar():
    return sqlite3.connect(DB)

def crear_tablas():
    with conectar() as con:
        con.execute('''CREATE TABLE IF NOT EXISTS pdfs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre_archivo TEXT NOT NULL,
            resumen TEXT)''')
        con.execute('''CREATE TABLE IF NOT EXISTS analisis_pdfs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            pdf_id INTEGER REFERENCES pdfs(id),
            fecha TEXT,
            num_paginas INTEGER,
            num_palabras INTEGER,
            num_caracteres INTEGER,
            palabras_clave TEXT,
            titulo TEXT,
            autor TEXT)''')

crear_tablas()
print("Tablas creadas: pdfs y analisis_pdfs")

## b. Procesamiento de PDF

In [ ]:
import os, re, shutil, datetime
from collections import Counter
from pypdf import PdfReader

os.makedirs("pdfs", exist_ok=True)

STOPWORDS = set('''de la que el en y a los del se las por un para con no una su al lo como más pero sus le ya o este
sí porque esta entre cuando muy sin sobre también me hasta hay donde quien desde todo nos durante todos uno les ni
contra otros ese eso ante ellos e esto mí antes algunos qué unos yo otro otras otra él tanto esa estos mucho quienes
nada muchos cual poco ella estar estas algunas algo nosotros mi mis tú te ti tu tus es son fue era soy lo he ha han
the and was is it to of for with this that are i my have not but you very from which their been has can will'''.split())

def palabras_utiles(texto):
    return [p for p in re.findall(r"[a-záéíóúñü]+", texto.lower()) if len(p) > 3 and p not in STOPWORDS]

def extraer_texto(ruta):
    lector = PdfReader(ruta)
    paginas = [(p.extract_text() or "") for p in lector.pages]
    meta = lector.metadata
    titulo = (meta.title or "") if meta else ""
    autor = (meta.author or "") if meta else ""
    return "\n".join(paginas), len(lector.pages), titulo, autor

def resumir(texto, n=3):
    texto = re.sub(r"-\s*\n\s*", "", texto)          # une palabras cortadas con guion al final de línea
    texto = re.sub(r"\s+", " ", texto).strip()
    oraciones = [o.strip() for o in re.split(r"(?<=[.!?])\s+", texto) if 40 <= len(o.strip()) <= 400]
    if not oraciones:
        return texto[:300]
    if len(oraciones) <= n:
        return " ".join(oraciones)
    freq = Counter(palabras_utiles(texto))
    def puntaje(o):
        ps = palabras_utiles(o)
        return sum(freq[p] for p in ps) / (len(ps) + 1)
    mejores = sorted(sorted(range(len(oraciones)), key=lambda i: puntaje(oraciones[i]), reverse=True)[:n])
    return " ".join(oraciones[i] for i in mejores)

def procesar_pdf(ruta):
    texto, n_pag, titulo, autor = extraer_texto(ruta)
    if not texto.strip():
        raise ValueError("No se pudo extraer texto (puede ser un PDF escaneado, solo con imágenes).")
    resumen = resumir(texto)
    n_pal = len(re.findall(r"\w+", texto))
    claves = ", ".join(p for p, _ in Counter(palabras_utiles(texto)).most_common(8))
    nombre = os.path.basename(ruta)
    shutil.copy(ruta, os.path.join("pdfs", nombre))
    ahora = datetime.datetime.now().isoformat(timespec="seconds")
    with conectar() as con:
        cur = con.execute("INSERT INTO pdfs (nombre_archivo, resumen) VALUES (?,?)", (nombre, resumen))
        con.execute(
            "INSERT INTO analisis_pdfs (pdf_id, fecha, num_paginas, num_palabras, num_caracteres, palabras_clave, titulo, autor) "
            "VALUES (?,?,?,?,?,?,?,?)",
            (cur.lastrowid, ahora, n_pag, n_pal, len(texto), claves, titulo, autor))
    return {"archivo": nombre, "paginas": n_pag, "palabras": n_pal, "palabras_clave": claves, "resumen": resumen}

## PDF de prueba (opcional)

In [ ]:
from fpdf import FPDF

TEXTOS = {
    "energia_solar.pdf": (
        "La energía solar aprovecha la radiación del sol para producir electricidad o calor. "
        "Los paneles fotovoltaicos convierten la luz en corriente eléctrica mediante celdas de silicio. "
        "En los últimos años el costo de los paneles solares bajó de forma notable en todo el mundo. "
        "Una instalación solar doméstica puede reducir de manera importante la factura de electricidad. "
        "Además, la energía solar no emite gases contaminantes durante su funcionamiento. "
        "Sin embargo, la producción de energía solar depende del clima y de las horas de luz del día. "
        "Por eso muchas instalaciones incorporan baterías para almacenar la energía sobrante. "
        "Los países con mucha radiación solar tienen una gran oportunidad para expandir esta tecnología."),
    "inteligencia_artificial.pdf": (
        "La inteligencia artificial es un campo de la informática que busca crear sistemas capaces de aprender. "
        "Las redes neuronales son modelos inspirados en el cerebro que aprenden patrones a partir de datos. "
        "Hoy la inteligencia artificial se usa en la medicina, la educación, el transporte y el comercio. "
        "Los modelos de lenguaje pueden traducir textos, resumir documentos y responder preguntas. "
        "El entrenamiento de estos modelos requiere grandes cantidades de datos y mucha capacidad de cómputo. "
        "También existen debates sobre los sesgos, la privacidad y el impacto de la inteligencia artificial en el empleo. "
        "Por esa razón es importante usar estas herramientas con criterio y supervisión humana."),
}
for nombre, texto in TEXTOS.items():
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Helvetica", size=12)
    pdf.multi_cell(0, 8, texto)
    pdf.output(nombre)
print("Creados:", ", ".join(TEXTOS), "(descárgalos del panel de archivos para subirlos en la interfaz)")

## c. Interfaz (Gradio)

In [ ]:
import gradio as gr

def ui_procesar(rutas):
    if not rutas:
        return pd.DataFrame(), "Sube uno o más archivos PDF."
    if isinstance(rutas, str):
        rutas = [rutas]
    filas, errores = [], []
    for r in rutas:
        try:
            res = procesar_pdf(r)
            filas.append({"Archivo": res["archivo"], "Páginas": res["paginas"], "Palabras": res["palabras"],
                          "Palabras clave": res["palabras_clave"], "Resumen": res["resumen"]})
        except Exception as e:
            errores.append(f"{os.path.basename(r)}: {e}")
    estado = f"PDF procesados y guardados: {len(filas)} de {len(rutas)}."
    if errores:
        estado += "\nErrores:\n" + "\n".join(errores)
    return pd.DataFrame(filas), estado

def ver_tablas():
    with conectar() as con:
        p = pd.read_sql_query("SELECT * FROM pdfs ORDER BY id DESC", con)
        a = pd.read_sql_query("SELECT * FROM analisis_pdfs ORDER BY id DESC", con)
    return p, a

with gr.Blocks(title="Procesamiento de PDF") as demo:
    gr.Markdown("# Procesamiento de PDF: resumen y análisis")
    with gr.Tabs():
        with gr.TabItem("Procesar PDF"):
            archivos = gr.File(file_count="multiple", file_types=[".pdf"], type="filepath", label="Archivos PDF")
            btn = gr.Button("Procesar y guardar", variant="primary")
            estado = gr.Textbox(label="Estado", lines=3)
            tabla = gr.Dataframe(label="Resultados")
            btn.click(ui_procesar, archivos, [tabla, estado])
        with gr.TabItem("Base de datos"):
            b = gr.Button("Actualizar tablas")
            t_p = gr.Dataframe(label="Tabla: pdfs")
            t_a = gr.Dataframe(label="Tabla: analisis_pdfs")
            b.click(ver_tablas, None, [t_p, t_a])

demo.launch(share=True, debug=True)

## Guardar la base de datos
En Colab los archivos se borran al cerrar la sesión. Descarga `pdfs.db`, o móntala en Google Drive:
```python
from google.colab import drive
drive.mount('/content/drive')
```
y cambia `DB = "pdfs.db"` por una ruta dentro de Drive.